# Quan sát profiler cho KV cache

Notebook này giải thích cách thu thập profiler cho một lần gọi public greedy generation. Agent chưa chạy notebook, capture mô hình thật, benchmark hay sweep; không có kết quả đo được lưu trong notebook này.

Mỗi capture thành công tự xuất `trace.json`, `regions.csv`, `operators.csv`, `manifest.json`, `report.ipynb` và các SVG graph phù hợp trong thư mục riêng của run. Báo cáo được render offline từ đúng artifact của run đó, không rerun model và không cần khởi chạy notebook kernel.

- Cached decode tái sử dụng K/V lịch sử; prefill vẫn xử lý toàn bộ prompt
- Controlled replay dùng continuation IDs cố định để cô lập decode, không gồm selector và chuẩn bị prompt/cache
- Actual generation gọi public wrapper, gồm selector, append token IDs và logic vòng lặp
- Repeated wall-time benchmark riêng dùng warm-up/lặp và đồng bộ ở ranh giới; không suy latency bình thường từ profiler đang active


## Metric trả lời câu hỏi gì

| Metric | Mục đích | Cách đọc và giới hạn |
| --- | --- | --- |
| CPU self / inclusive total (`us`) | Thời gian profiler quy cho scope và phần con | CPU giúp thấy dispatch, Python loop và scalar waits; parent total đã chứa child, không cộng hàng cha-con |
| Device self / total (`us`) | Thời gian device mà profiler gán cho op | CUDA bất đồng bộ; CPU/device có thể overlap, không cộng chúng thành latency |
| Operator/kernel count và shapes | Thấy op/kernel nào xuất hiện và input shapes | Count/shape lớn tự nó không chứng minh bottleneck |
| Memory bytes khi bật | Allocation/free accounting của profiler | Không phải HBM/GDDR-to-SM traffic hoặc memory bandwidth |

Capture đồng bộ selected CUDA device tại ranh giới trước/sau một wrapper call, không sync theo từng op hoặc token. Selector có thể gây scalar wait trên CPU; đây là giả thuyết để đọc event, không mặc định là bottleneck.

Một trace chỉ mô tả workload đó, không xác lập quan hệ nhân quả hay xu hướng tổng quát. Profiler duration không phải ordinary wall-clock latency.

Tài liệu: [PyTorch Profiler](https://pytorch.org/docs/stable/profiler.html), [PyTorch CUDA semantics](https://pytorch.org/docs/stable/notes/cuda.html), [NVIDIA Nsight Compute Memory Workload Analysis](https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#memory-workload-analysis). Không cài hoặc chạy Nsight trong notebook này.


## Semantic scope có thể quan sát bên ngoài core

Adapter bọc tạm method/module instance thật và dọn wrapper khi thoát scope. Không thay đổi source core, không patch class/module global/torch và không sao chép phép tính. Scope chỉ đặt tên cho biên callable/module hiện có.

```text
generation
  prefill                         # Cached prefill method / uncached forward đầu tiên
    embedding                    # InputEmbedding module call
    layer/0 ... layer/L-1        # Block call hoặc cached block method
      ln1
      attention
        head/0 ... head/H-1    # Chỉ ops, head callable thật
      ln2 -> ffn
    final_norm -> lm_head
  selector/0                     # Toàn callable selector hiện có
  decode/1                       # Cached method, xử lý token được chọn đầu tiên
  prefix_forward/1               # Uncached full-prefix forward
  selector/1 ...
```

Không có semantic label riêng cho residual addition, append-ID/cache `cat`, Q/K/V projection, attention score/softmax hay thành phần selector bên trong. Các phép toán đó vẫn có native PyTorch operator events; cached decode embedding nằm trong scope `decode/j` vì token lookup và cộng position embedding không có biên module chung để bọc. `head/h` chỉ có ở mức `ops`; không bịa label khi thiếu boundary.

| Tensor | Prefill/full-prefix (`R=P`, `K=P`) | One-token cached decode (`R=1`, `K=T_old+1`) |
| --- | --- | --- |
| Model input IDs | `token_ids [P]` | `token_ids_new [1]` |
| Hidden states | `[P,D]` | `[1,D]` |
| Q | `[P,d_k]` | `[1,d_k]` |
| K / V cache | `[P,d_k]` / `[P,d_v]` | `[T_old+1,d_k]` / `[T_old+1,d_v]` |
| Scores | `[P,P]` | `[1,T_old+1]` |
| Attention weights | `[H,P,P]` | `[H,1,T_old+1]` |
| Head output / concatenated output | `[P,d_v]` / `[P,H*d_v]` | `[1,d_v]` / `[1,H*d_v]` |
| Logits | `[P,V]` | `[1,V]` |

`R` là query rows, `K` là key context. Với effective budget `N`, wrapper append N token IDs nhưng chỉ xử lý tiếp N-1 token qua model; cached length cuối `P+N-1`. Uncached `prefix_forward/j` nhận prefix có length `P+j`.

Workload nhỏ để học cách đọc trace: `P=8` và `P=96`, `B=4`, hai arm cached/uncached. Đây không phải historical generation `B=16` và chưa được đo.


In [7]:
from pathlib import Path
import sys

candidates = (Path.cwd(), *Path.cwd().parents)
ROOT = next((path for path in candidates if (path / 'llm_foundations/experiments/kv_cache_benchmark.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Chạy notebook từ repo root hoặc llm_foundations/experiments')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from llm_foundations.experiments.kv_cache_benchmark import BenchmarkConfig, ModelConfig
from llm_foundations.experiments.kv_cache_profiling import ProfileConfig, capture_profile
from llm_foundations.experiments.kv_cache_profiling_report import render_profile_report
from IPython.display import FileLink, display


## Cấu hình và case matrix

`ModelConfig()` giữ baseline learner hiện có. Setup chỉ import API và tạo cấu hình, không capture. Device `auto` ưu tiên CUDA; CPU chỉ được chọn khi CUDA không khả dụng. Nếu CUDA được chọn mà capture lỗi, harness báo lỗi, không retry CPU.


In [8]:
MODEL = ModelConfig()
BASE_CONFIG = BenchmarkConfig(
    model=MODEL, seed=20261023, dtype='float32', device='auto',
    warmup=0, repeats=1, prompt_lengths=(8,), generated_tokens=4,
    generation_lengths=(4,), generation_prompt_length=8,
)
CASE_MATRIX = [(arm, prompt_length, 4) for prompt_length in (8, 96) for arm in ('cached', 'uncached')]
OUTPUT_ROOT = ROOT / 'llm_foundations/experiments/kv_cache_profiling_runs'
CASE_MATRIX


[('cached', 8, 4), ('uncached', 8, 4), ('cached', 96, 4), ('uncached', 96, 4)]

## Capture có opt-in rõ ràng

Capture mặc định tắt. Bật `RUN_PROFILE` chỉ khi mày muốn tự thực hiện workload. Mỗi lần capture tạo report notebook và graphs của run đó ngay sau khi export; model không bị chạy lại để render report. `RUN_OPS`, `RUN_MEMORY`, `RUN_STACKS` là các pass riêng, không gộp shapes, allocation và stack vào cùng một capture.

Khi run xong, cell tạo link tới đúng `report.ipynb`. Click link trong Jupyter để mở report; có thể gọi `render_profile_report(run_path)` offline lần nữa nếu chỉ muốn tái tạo report từ artifact đã có.


In [9]:
RUN_PROFILE = True
RUN_OPS = False
RUN_MEMORY = False
RUN_STACKS = False
BLOCK_RUNS = {}
OPS_RUNS = {}
MEMORY_RUNS = {}
STACK_RUNS = {}

if RUN_PROFILE:
    for arm, prompt_length, budget in CASE_MATRIX:
        profile = ProfileConfig(arm=arm, prompt_length=prompt_length, max_new_tokens=budget, level='blocks', warmup=2)
        BLOCK_RUNS[(arm, prompt_length)] = capture_profile(BASE_CONFIG, profile, output_root=OUTPUT_ROOT)
    if RUN_OPS:
        for arm, prompt_length, budget in CASE_MATRIX:
            profile = ProfileConfig(arm=arm, prompt_length=prompt_length, max_new_tokens=budget, level='ops', warmup=2, record_shapes=True)
            OPS_RUNS[(arm, prompt_length)] = capture_profile(BASE_CONFIG, profile, output_root=OUTPUT_ROOT)
    if RUN_MEMORY:
        for arm, prompt_length, budget in CASE_MATRIX:
            profile = ProfileConfig(arm=arm, prompt_length=prompt_length, max_new_tokens=budget, level='blocks', warmup=2, profile_memory=True)
            MEMORY_RUNS[(arm, prompt_length)] = capture_profile(BASE_CONFIG, profile, output_root=OUTPUT_ROOT)
    if RUN_STACKS:
        for arm, prompt_length, budget in CASE_MATRIX:
            profile = ProfileConfig(arm=arm, prompt_length=prompt_length, max_new_tokens=budget, level='blocks', warmup=2, with_stack=True)
            STACK_RUNS[(arm, prompt_length)] = capture_profile(BASE_CONFIG, profile, output_root=OUTPUT_ROOT)

    for run_path in BLOCK_RUNS.values():
        display(FileLink(str(run_path / 'report.ipynb')))
else:
    print('Capture đang tắt; report sẽ được tạo tự động cho từng run khi RUN_PROFILE được bật')

/home/coronny/rag-project/llm_foundations/experiments/kv_cache_profiling_runs/20261025T121558Z_7ebf1751/report.ipynb

/home/coronny/rag-project/llm_foundations/experiments/kv_cache_profiling_runs/20261025T121559Z_1c5d6a68/report.ipynb

/home/coronny/rag-project/llm_foundations/experiments/kv_cache_profiling_runs/20261025T121600Z_650a5c23/report.ipynb

/home/coronny/rag-project/llm_foundations/experiments/kv_cache_profiling_runs/20261025T121601Z_576954b3/report.ipynb

## Bảng và graph trong report

`regions.csv` giữ ancestry; report hiển thị CPU self/inclusive total từng region và graph semantic leaves để tránh cộng parent-child inclusive totals. `operators.csv` được xếp theo CPU self. Device/kernel graph chỉ xuất hiện nếu run thực sự có metric device hoặc trace kernel; CPU-only/no-kernel được ghi rõ. Null có reason, số zero thật vẫn zero. Không cộng CPU/device overlap và không gọi profiler time là normal latency.

Report được tạo từ `manifest.json`, hai CSV và trace của cùng thư mục run. Không chọn file từ run khác, không mở kernel, không gọi model. SVG nằm trong `graphs/` và được nhúng vào notebook report.

*Note* (Semantic region table):
1. Region: Vùng đâng tính toán hiện tại
2. Ancestry: Thuộc nhánh cha nào trong cây tính toán
3. Count: Số lần thực hiện phép tính của vùng hiện tại


In [10]:
OPEN_RUN = next(iter(BLOCK_RUNS.values()), None)
if OPEN_RUN is not None:
    report_path = render_profile_report(OPEN_RUN)
    display(FileLink(str(report_path)))
else:
    print('Chưa có run mới; không đọc hoặc chọn run lịch sử tự động')

/home/coronny/rag-project/llm_foundations/experiments/kv_cache_profiling_runs/20261025T121558Z_7ebf1751/report.ipynb

## Phạm vi và giới hạn

- Instrumentation chỉ bọc instance method/module boundaries tạm thời; cùng model không được chạy đồng thời trong lúc adapter đang bật
- Không có scope riêng cho residual, append/cat, projection, softmax hay selector nội bộ; dùng native operator trace thay vì semantic label giả
- Profiler overhead khiến trace không phù hợp làm wall-time benchmark
- Memory profiler ghi allocation/free, không đo memory traffic/bandwidth
- Một trace không chứng minh bottleneck, nguyên nhân hay hành vi agent production

Learner mới chạy case sau; inspect capture status, per-run report và metric limits trước khi diễn giải.
